In [1]:
from ETGEMs_function_PMI import *
import pandas as pd
import cobra
import ast
from cobra.io import write_sbml_model
from numpy import *
import copy

In [2]:
#Get Reaction G0 from local file_original
reaction_g0_file_original0 = './reaction_g0_strain0.txt'
reaction_g0_file_original1 = './reaction_g0_strain1.txt'
#Get Metabolite concentration from local file
metabolites_lnC_file_original0 = './metabolites_lnC_strain0.txt'
metabolites_lnC_file_original1 = './metabolites_lnC_strain1.txt'
#Get Model from local file
model_file_original0 = './iML1515.xml'
model_file_original1 = './iMM904.xml'
#Get reaction kcat data from ECMpy
reaction_kcat_MW_file_original0 = './ID_kcat_MW_file_strain0.csv'
reaction_kcat_MW_file_original1 = './ID_kcat_MW_file_strain1.csv'

In [3]:
## Convert to usable model formats
model0=Get_Concretemodel_Need_Data(reaction_g0_file_original0,metabolites_lnC_file_original0,model_file_original0,reaction_kcat_MW_file_original0)
model1=Get_Concretemodel_Need_Data(reaction_g0_file_original1,metabolites_lnC_file_original1,model_file_original1,reaction_kcat_MW_file_original1)
## knockout the fhuA gene
model0['lb_list']['EX_fecrm_e ']=0
model0['lb_list']['EX_fecrm_e ']=0

In [4]:
# get the information of km, vmax and public metabolites
km = pd.read_csv('./km.csv')
vmax = pd.read_csv('./vmax.csv')
public_metabolism = pd.read_csv('./public_metabolism_test.csv')

'''
public_metabolism_name_list = []
public_metabolism_concentration_list = []
for i in public_metabolism['metabolism']:
    public_metabolism_name_list.append(i)
for j in public_metabolism['concentration']:
    public_metabolism_concentration_list.append(j)
public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
print(public_metabolism)
'''

"\npublic_metabolism_name_list = []\npublic_metabolism_concentration_list = []\nfor i in public_metabolism['metabolism']:\n    public_metabolism_name_list.append(i)\nfor j in public_metabolism['concentration']:\n    public_metabolism_concentration_list.append(j)\npublic_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))\nprint(public_metabolism)\n"

In [5]:
# definate the function to simulate metabolism and interaction
def time_space_state(model_list, biomass_list, growth_list, breed_list, parameter_list, public_metabolism, vmax, km, deta_t, deta_s, micro_distribute_threshold, length, D):
    
    number_cell_side = length // deta_s
    number_cell_side = int(number_cell_side)
    public_metabolism_name_list = []
    public_metabolism_concentration_list = []
    for i in public_metabolism['metabolism']:
        public_metabolism_name_list.append(i)
    for j in public_metabolism['concentration']:
        public_metabolism_concentration_list.append(j)
    public_metabolism = dict(zip(public_metabolism_name_list, public_metabolism_concentration_list))
    
    number_model = len(model_list)
    
    k_m = {}
    v_max = {}
    for i in range(number_model):
        for j in range(len(km['reactions_strain'+str(i)])):
            k_m[(i, km['reactions_strain'+str(i)][j])] = km['km_strain'+str(i)][j]
            v_max[(i, vmax['reactions_strain'+str(i)][j])] = vmax['vmax_strain'+str(i)][j]
    
    
    number_public_metabolism = len(public_metabolism)
    distribute_micro_list = {}
    distribute_public_metabolism_list = {}
    distribute_lb_list = {}
    public_metabolism_r_list = []
    #set the initial distribution of straints
    for i in range(number_model):
        distribute_micro = np.random.randint(40, size=number_cell_side)
        distribute_micro_list.update({i: distribute_micro})
    print(distribute_micro_list)
    #set the initial distribution of metabolites
    for i in public_metabolism_name_list:
        distribute_public_metabolism = multiply(np.mat(np.ones(number_cell_side)), public_metabolism[i])
        distribute_public_metabolism_list.update({i: distribute_public_metabolism})
    #calculate the upperbounds of uptake reactions for public metabolites
    public_reaction_ub_list = {}
    public_reaction_list = {}
    for i in range(number_model):
        public_reaction_ub = {}
        public_reaction = []
        for rea in model_list[i]['reaction_list']:
            if 'EX_' in rea:
                for j in public_metabolism_name_list:
                    try:
                        model_list[i]['coef_matrix'][(j,rea)]
                    except:
                        pass
                    else:
                        ub = np.mat(np.ones(number_cell_side))
                        if model_list[i]['coef_matrix'][(j,rea)] > 0:
                            for m in range(number_cell_side):
                                ub[0,m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                        else:
                            try:
                                model_list[i]['ub_list'][rea]
                            except:
                                ub = ub * 1000
                            else:
                                ub = ub * model_list[i]['ub_list'][rea]
                        public_reaction_ub.update({rea: ub})
                        public_reaction.append(rea)
                        break
            public_reaction_ub_list[i] = public_reaction_ub
            public_reaction_list[i] = public_reaction

            
    ct = 0
    
    distribute_micro_list_t = {ct: distribute_micro_list}
    distribute_public_metabolism_list_t = {ct: distribute_public_metabolism_list}
    distribute_lb_list_t = {ct: distribute_lb_list}
    r = micro_distribute_threshold + 1
    r_threshold = [r]*5
    slip_r = np.mean(r_threshold[-5:])
    number = {}
    various = {}
    for i in range(number_model):
        number[i] = [np.mean(distribute_micro_list[i])]
        various[i] = [np.std(distribute_micro_list[i])/np.mean(distribute_micro_list[i])]
    
    
    # iterative simulation by slip_r
    while slip_r >= micro_distribute_threshold:
        ct += 1
        print(ct)
        distribute_micro_list_t[ct] = copy.deepcopy(distribute_micro_list_t[ct-1])
        micro_decrease = {}
        micro_increase = {}
        
        #simulate the cell wandering
        #micro_decrease: the decrease number of strains
        #micro_increase: the increase number of strains
        #micro_decrease_cell: the decrease number of strains in the current grid
        #micro_increase_fcell: the increase number of strains in the forward grid
        #micro_increase_bcell: the increase number of strains in the backward grid
        for m in range(number_cell_side):
            micro_decrease_cell = {}
            micro_increase_cell = {}
            for i in range(number_model):
                micro_decrease_cell[i] = 0
                micro_increase_cell[i] = 0
            micro_decrease[m] = micro_decrease_cell
            micro_increase[m] = micro_increase_cell
            
            
        if ct > 0:
            met = 'glc__D_e'
            threshold = 0.3
            for m in range(number_cell_side):
                #calculate the number of strains in the internal grids
                if 0<m<number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = (distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)+distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-2*distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * micro_decrease_cell
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_increase_bcell = micro_decrease_cell-micro_increase_fcell
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                                    micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                                else:
                                    micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_decrease_cell = int(micro_decrease_cell)
                                    micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                    micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                    micro_increase_fcell = int(micro_increase_fcell)
                                    micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                    micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
                            elif distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the first grid
                elif m == 0:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m+1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_bcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m+1]/(distribute_micro_list_t[ct][i][m+1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_bcell = micro_increase_bcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_bcell = int(micro_increase_bcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m+1][i] = micro_increase[m+1][i] + micro_increase_bcell
                #calculate the number of strains in the last grid
                elif m == number_cell_side-1:
                    for i in range(number_model):
                        if distribute_micro_list_t[ct][i][m] > 0:
                            if distribute_public_metabolism_list[met][0,m-1] > distribute_public_metabolism_list[met][0,m]:
                                micro_decrease_cell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_decrease_cell = micro_decrease_cell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_decrease_cell = int(micro_decrease_cell)
                                micro_increase_fcell_ratio = min(threshold,(distribute_public_metabolism_list[met][0,m-1]/(distribute_micro_list_t[ct][i][m-1]+0.001)-distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001))/(distribute_public_metabolism_list[met][0,m]/(distribute_micro_list_t[ct][i][m]+0.001)+0.001))
                                micro_increase_fcell = micro_increase_fcell_ratio * distribute_micro_list_t[ct][i][m]
                                micro_increase_fcell = int(micro_increase_fcell)
                                micro_decrease[m][i] = micro_decrease[m][i] + micro_decrease_cell
                                micro_increase[m-1][i] = micro_increase[m-1][i] + micro_increase_fcell
        
            #calculate the number of strains after wandering
            for m in range(number_cell_side):
                for i in range(number_model):
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] - micro_decrease[m][i]
                    distribute_micro_list_t[ct][i][m] = distribute_micro_list_t[ct][i][m] + micro_increase[m][i]
                    distribute_micro_list_t[ct][i][m] = max(0, distribute_micro_list_t[ct][i][m])
                            
                            
        
        #simulate the substrate diffusion by Fick's second law
        for m in range(number_cell_side): 
            if 0<m<number_cell_side-1:
                for met in public_metabolism_name_list:
                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + ((distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s-(distribute_public_metabolism_list[met][0,m]-distribute_public_metabolism_list[met][0,m-1])/deta_s)/deta_s*D*deta_t
                    if distribute_public_metabolism_list[met][0,m] < 0:
                        print('Warning: the D is too high!')
                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == 0:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m+1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            if m == number_cell_side-1:
                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + (distribute_public_metabolism_list[met][0,m-1]-distribute_public_metabolism_list[met][0,m])/deta_s/deta_s*D*deta_t
                distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
            
        #simulate the metabolism by ETMs
        biomass_value_list = {}
        number_model_range = []
        for m in range(number_cell_side):
            if m%2 == 0:
                number_model_range.append([0,1])
            elif m%2 == 1:
                number_model_range.append([1,0])
        for m in range(number_cell_side):
            B_value_list = []
            
            
            biomass_value_micro = {}
            for i in number_model_range[m]:
                if distribute_micro_list_t[ct][i][m] > 0:
                    public_metabolism_flux_list = {}
                    for j in public_metabolism_name_list:
                        public_metabolism_flux_list.update({j: 0})
                    for j in public_reaction_list[i]:
                        model_list[i]['ub_list'][j] = int(public_reaction_ub_list[i][j][0,m])
            
                    biomass_id = biomass_list[i]
                    E_total=parameter_list[i][0]
                    #set the carbon source to glucose
                    substrate_name='EX_glc__D_e_reverse'
                    substrate_value=parameter_list[i][1]
                    biomass_value=growth_list[i]
                    K_value=parameter_list[i][2]

                    try:
                        MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                    except:
                        save_rate = 0.9
                        while save_rate >= 0:
                            biomass_value2 = save_rate * biomass_value
                            try:
                                MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                            except:
                                save_rate = save_rate - 0.1
                            else:
                                #calculate the MDF values of metabolic networks
                                biomass_value_micro[i] = biomass_value2
                                B_value2 = MDF_Calculation(model_list[i],biomass_value2,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                                #calculate the biomass yield under the MDF value
                                obj_name=biomass_list[i]
                                obj_target='maximize'
                                if i == 0:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                elif i == 1:
                                    max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value2,'gurobi')
                                biomass_value=max_biomass_under_mdf*0.9
                                
                                #calculate the minimum value of the sum of the fluxes
                                if i == 0:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                elif i == 1:
                                    [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                                
                                #translating the results of ETMs into a usable form
                                model=model_list[i]['model']
                                reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                                reaction_g0=model_list[i]['reaction_g0']
                                coef_matrix=model_list[i]['coef_matrix']
                                metabolite_list=model_list[i]['metabolite_list']
                                use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
                                distribute_public_metabolism_increase = {}
                                distribute_public_metabolism_decrease = {}
                                
                                #simulate the fluxes of the public metabolites
                                for rea in public_reaction_list[i]:
                                    for met in public_metabolism_name_list:
                                        try:
                                            model_list[i]['coef_matrix'][(met,rea)]
                                        except:
                                            pass
                                        else:
                                            public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
                                
                                #simulate the distribution of the public metabolites
                                for met in public_metabolism_name_list:
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]*save_rate
                                    distribute_public_metabolism_list[met][0,m] = max(0,distribute_public_metabolism_list[met][0,m])
                                    
                                    
                                save_rate_ub = [save_rate]
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        met = rea[3::]
                                        if met in public_metabolism_name_list:
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            save_rate_met = distribute_public_metabolism_list[met][0,m]/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                            save_rate_ub.append(save_rate_met)
                                        
                                save_rate_final = min(save_rate_ub)
                                
                                for rea in public_reaction_list[i]:
                                    if 'reverse' not in rea:
                                        if met in public_metabolism_name_list:
                                            met = rea[3::]
                                    
                                            intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                            intracellular_c = intracellular_c.replace(';',',"')
                                            intracellular_c = intracellular_c.replace(' :','" :')
                                            intracellular_c = '{"' + intracellular_c + '}'
                                            intracellular_c_dict = ast.literal_eval(intracellular_c)
                                        
                                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - intracellular_c_dict[met]*int((save_rate_final)*distribute_micro_list_t[ct][i][m]) + intracellular_c_dict[met]*int((1-save_rate_final)*distribute_micro_list_t[ct][i][m])
                                            if distribute_public_metabolism_list[met][0,m] < 0:
                                                print(met + '_3: ', distribute_public_metabolism_list[met][0,m])
                                    
                                
                                distribute_micro_decrease = int((1-save_rate_final) * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_increase = int(save_rate_final * distribute_micro_list_t[ct][i][m])/breed_list[i]
                                distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] - distribute_micro_decrease + distribute_micro_increase)
                                
                                break
                            
                        continue
                    else:
                        #calculate the MDF values of metabolic networks
                        biomass_value_micro[i] = biomass_value
                        B_value=MDF_Calculation(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,'gurobi')
                        B_value_list.append(B_value)
                        #calculate the biomass yield under the MDF value
                        obj_name=biomass_list[i]
                        obj_target='maximize'
                        if i == 0:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation0(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            max_biomass_under_mdf=Max_Growth_Rate_Calculation1(model_list[i],obj_name,obj_target,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        biomass_value=max_biomass_under_mdf*0.9
                        
                        #calculate the minimum value of the sum of the fluxes
                        if i == 0:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation0(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
                        elif i == 1:
                            [min_V,Concretemodel]=Min_Flux_Sum_Calculation1(model_list[i],biomass_value,biomass_id,substrate_name,substrate_value,K_value,E_total,B_value,'gurobi')
            
                        #translating the results of ETMs into a usable form
                        model=model_list[i]['model']
                        reaction_kcat_MW=model_list[i]['reaction_kcat_MW']
                        reaction_g0=model_list[i]['reaction_g0']
                        coef_matrix=model_list[i]['coef_matrix']
                        metabolite_list=model_list[i]['metabolite_list']
                        use_result = Get_Results_Thermodynamics(model,Concretemodel,reaction_kcat_MW,reaction_g0,coef_matrix,metabolite_list)
                        
                        #simulate the fluxes of the public metabolites
                        for rea in public_reaction_list[i]:
                            for met in public_metabolism_name_list:
                                try:
                                    model_list[i]['coef_matrix'][(met,rea)]  
                                except:
                                    pass
                                else:
                                    public_metabolism_flux_list[met] = public_metabolism_flux_list[met] + model_list[i]['coef_matrix'][(met,rea)]* use_result['flux'][rea]
            
                        #simulate the distribution of the public metabolites
                        distribute_public_metabolism_ori = copy.deepcopy(distribute_public_metabolism_list)
                        distribute_public_metabolism_re = copy.deepcopy(distribute_public_metabolism_ori)
                        for met in public_metabolism_name_list:
                            distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] - public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m]
                            distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                            if distribute_public_metabolism_list[met][0,m] < 0:
                                print(met+'_1: ', distribute_public_metabolism_list[met][0,m])
                                
                        #simulate the multiplication and death rates of strains and the distribution of public metabolites after multiplication or death                
                        death_rate = 0
                        birth_rate = 1
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    bd_rate_met = (distribute_public_metabolism_list[met][0,m] - 0.1)/intracellular_c_dict[met]/distribute_micro_list_t[ct][i][m]
                                    if bd_rate_met < 0:
                                        death_rate_lb = (public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] - distribute_public_metabolism_re[met][0,m] + 0.1)/(public_metabolism_flux_list[met] * deta_t * distribute_micro_list_t[ct][i][m] + intracellular_c_dict[met] * distribute_micro_list_t[ct][i][m])
                                        death_rate = max(death_rate, death_rate_lb)
                                    else:
                                        birth_rate = min(birth_rate, bd_rate_met)
                        death_rate = min(death_rate, 1)
                        birth_rate = min(birth_rate,1)
                        if death_rate > 0:
                            birth_rate = 0
                            for met in public_metabolism_name_list:
                                distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_list[met][0,m] + public_metabolism_flux_list[met] * deta_t * math.ceil(distribute_micro_list_t[ct][i][m] * death_rate)
                                distribute_public_metabolism_ori[met][0,m] = copy.deepcopy(distribute_public_metabolism_list[met][0,m])
                        
                        for rea in public_reaction_list[i]:
                            if 'reverse' not in rea:
                                met = rea[3::]
                                if met in public_metabolism_name_list:
                            
                                    intracellular_c = use_result['met_concentration'][rea].lstrip(';')
                                    intracellular_c = intracellular_c.replace(';',',"')
                                    intracellular_c = intracellular_c.replace(' :','" :')
                                    intracellular_c = '{"' + intracellular_c + '}'
                                    intracellular_c_dict = ast.literal_eval(intracellular_c)
                                
                                    distribute_public_metabolism_list[met][0,m] = distribute_public_metabolism_ori[met][0,m] - intracellular_c_dict[met]*math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i]) + intracellular_c_dict[met]*math.ceil(death_rate*distribute_micro_list_t[ct][i][m])
                                    if distribute_public_metabolism_list[met][0,m] < 0:
                                        print(met+'_2: ', distribute_public_metabolism_list[met][0,m])
                                        distribute_public_metabolism_list[met][0,m] = 0
                        
                        #simulate the distribution of strains after multiplication or death
                        distribute_micro_increase = math.floor(distribute_micro_list_t[ct][i][m]*birth_rate/breed_list[i])
                        distribute_micro_decrease = math.ceil(distribute_micro_list_t[ct][i][m]*death_rate)
                        distribute_micro_list_t[ct][i][m] = max(0,distribute_micro_list_t[ct][i][m] + distribute_micro_increase - distribute_micro_decrease)
            
            #calculate the upperbounds of nutrient uptake rates after substrate diffusion, cell wandering, metabolism, multiplication and death
            biomass_value_list[m] = biomass_value_micro
            
            for i in range(number_model):
                for rea in model_list[i]['reaction_list']:
                    if 'EX_' in rea:
                        for j in public_metabolism_name_list:
                            try:
                                model_list[i]['coef_matrix'][(j,rea)]
                            except:
                                pass
                            else:
                                if model_list[i]['coef_matrix'][(j,rea)] > 0:
                                    public_reaction_ub_list[i][rea][0, m] = v_max[(i,rea)]*distribute_public_metabolism_list[j][0,m]/(distribute_public_metabolism_list[j][0,m]+k_m[(i,rea)])
                                    if i == 0:
                                        # gene upregulation for Fe3+ intake
                                        if 'EX_fe3_e' in rea:
                                            public_reaction_ub_list[i][rea][0, m] = public_reaction_ub_list[i][rea][0, m] * 1.2
        
        # calculate the mean number and uniformity of distribution of strains at this iteration
        for i in range(number_model):
            strain_number = np.mean(distribute_micro_list_t[ct][i])
            strain_various = np.std(distribute_micro_list_t[ct][i])
            number[i].append(strain_number)
            various[i].append(strain_various)
            print('strain_number: ', i, strain_number)
            print('strain_various: ', i, strain_various)
        
        # calculate the slip_r at this iteration
        if ct > 1:
            r = 0
            for i in range(number_model):
                for m in range(number_cell_side):
                    if distribute_micro_list_t[ct-1][i][m] > 0:
                        r = r + ((distribute_micro_list_t[ct][i][m]-distribute_micro_list_t[ct-1][i][m])/(distribute_micro_list_t[ct-1][i][m]))**2
                    else:
                        r = r + (distribute_micro_list_t[ct][i][m])**2
        
        r_threshold.append(r)
        slip_r = np.mean(r_threshold[-5:])
        fd_r_threshold = r_threshold[5:]
        print('fd_r_threshold: ', fd_r_threshold)
        print('slip_r: ', slip_r)
    return(distribute_micro_list_t, number, various)

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([34, 32, 21, 38, 27, 20,  1,  2, 22, 20,  7, 29, 33, 28,  0, 36, 12,
       22, 27]), 1: array([33, 39, 14,  0, 38, 27, 24, 13, 11, 10,  2, 11,  2, 10, 10, 28,  4,
       29, 26])}
1
strain_number:  0 25.57894736842105
strain_various:  0 13.944686931903808
strain_number:  1 17.57894736842105
strain_various:  1 12.444892376408395
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.263157894736842
strain_various:  0 11.511285562041223
strain_number:  1 17.63157894736842
strain_various:  1 8.82705486031016
fd_r_threshold:  [1.05, 786.5651872716608]
slip_r:  158.15303745433215
3
strain_number:  0 35.89473684210526
strain_various:  0 11.543486369478108
strain_number:  1 17.68421052631579
strain_various:  1 8.13085501802238
fd_r_threshold:  [1.05, 786.5651872716608, 9.440413690313326]
slip_r:  159.83112019239482
4
strain_number:  0 42.63157894736842
strain_various:  0 12.545540311897751
strain_number:  1 17.68421052631579
strain_various:  1 7.108576225636092
fd_r_threshold

glc__D_e_1:  -2.2377651306901614
glc__D_e_1:  -8.686659627114912
glc__D_e_1:  -12.74954445659616
glc__D_e_1:  -6.010152272928929
glc__D_e_1:  -3.9734953740757106
glc__D_e_1:  -10.477981624269614
glc__D_e_1:  -10.661847712373307
glc__D_e_1:  -2.7395518536418484
glc__D_e_1:  -1.5499124665932493
glc__D_e_1:  -5.048051851023107
glc__D_e_1:  -15.413147569756985
glc__D_e_1:  -3.684148203460591
glc__D_e_1:  -4.672013412961119
glc__D_e_1:  -15.110839812481544
glc__D_e_1:  -8.220055137034269
glc__D_e_1:  -3.9500183640669517
glc__D_e_1:  -4.794649610496594
glc__D_e_1:  -9.677076660215132
glc__D_e_1:  -5.9212695138795315
glc__D_e_1:  -3.9481350088971316
glc__D_e_1:  -6.026488591098696
glc__D_e_1:  -8.122839334336923
glc__D_e_1:  -4.521476609802117
strain_number:  0 14.421052631578947
strain_various:  0 5.860804592452654
strain_number:  1 6.368421052631579
strain_various:  1 2.53834853721842
fd_r_threshold:  [1.05, 786.5651872716608, 9.440413690313326, 6.895635008163561, 7.135478245850704, 6.46629

glc__D_e_1:  -1.1232703957166372
glc__D_e_1:  -0.9688259739158432
strain_number:  0 3.1052631578947367
strain_various:  0 1.0205641805087007
strain_number:  1 1.368421052631579
strain_various:  1 0.7405919620773836
fd_r_threshold:  [1.05, 786.5651872716608, 9.440413690313326, 6.895635008163561, 7.135478245850704, 6.466291269339385, 7.014655795262499, 4.899879113274264, 4.249133548814148, 5.2181658602517365, 4.857931473772869, 3.9285659027962794, 5.065855026294162, 3.5310692301518642, 4.678670319979842, 4.534326184177375, 6.591666666666667]
slip_r:  4.880317485453982
18
glc__D_e_1:  -0.07651197213653105
glc__D_e_1:  -0.073492113946662
glc__D_e_1:  -0.7435484003751482
glc__D_e_1:  -0.2762021728808164
glc__D_e_1:  -0.12175775108002229
glc__D_e_1:  -0.42934646817354527
glc__D_e_1:  -0.8501838083638935
glc__D_e_1:  -0.6957393865630995
glc__D_e_1:  -0.48645206134567576
glc__D_e_1:  -0.7857198496132628
glc__D_e_1:  -0.3894613992151983
glc__D_e_1:  -0.5794059477678886
glc__D_e_1:  -0.966661093

glc__D_e_1:  -0.08603162675167841
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 786.5651872716608, 9.440413690313326, 6.895635008163561, 7.135478245850704, 6.466291269339385, 7.014655795262499, 4.899879113274264, 4.249133548814148, 5.2181658602517365, 4.857931473772869, 3.9285659027962794, 5.065855026294162, 3.5310692301518642, 4.678670319979842, 4.534326184177375, 6.591666666666667, 4.739722222222222, 6.444444444444445, 3.7222222222222223, 5.861111111111111, 2.25, 6.5, 1.3611111111111112, 2.0, 3.25, 4.0]
slip_r:  3.422222222222222
28
glc__D_e_1:  -0.012078705941612755
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 786.5651872716608, 9.440413690313326, 6.895635008163561, 7.135478245850704, 6.466291269339385, 7.014655795262499, 4.899879113274264, 4.24913354881

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([26, 17, 38,  0,  0, 14,  6, 33, 14,  5, 27, 12, 15, 33, 16, 31,  9,
       22, 20]), 1: array([33,  9, 16,  9, 21, 17, 10, 38, 36, 26, 39, 35,  4,  1, 30, 36, 34,
        4, 35])}
1
strain_number:  0 21.0
strain_various:  0 13.12290083463978
strain_number:  1 23.263157894736842
strain_various:  1 13.509614920154412
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 24.736842105263158
strain_various:  0 8.539950112409187
strain_number:  1 23.31578947368421
strain_various:  1 8.535083211624997
fd_r_threshold:  [1.05, 366.2526672304941]
slip_r:  74.09053344609882
3
strain_number:  0 29.157894736842106
strain_various:  0 8.112778470735872
strain_number:  1 23.526315789473685
strain_various:  1 7.177218498412962
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847]
slip_r:  76.27197842181539
4
strain_number:  0 34.578947368421055
strain_various:  0 10.043671951053947
strain_number:  1 23.736842105263158
strain_various:  1 6.544118333629285
fd_r_threshold:  [1.05, 

glc__D_e_1:  -2.2230882488898183
glc__D_e_1:  -3.251547502153252
glc__D_e_1:  -9.650338613577217
glc__D_e_1:  -9.325321356218856
glc__D_e_1:  -4.698519846711628
glc__D_e_1:  -4.749416207940962
glc__D_e_1:  -9.10432922897609
glc__D_e_1:  -8.779311971617728
glc__D_e_1:  -3.560792218835134
glc__D_e_1:  -4.434807050297774
glc__D_e_1:  -9.410495221720142
glc__D_e_1:  -8.593685445695964
glc__D_e_1:  -3.6096345699588026
glc__D_e_1:  -3.8149753529889314
glc__D_e_1:  -5.3284548651289265
glc__D_e_1:  -4.947845854001414
glc__D_e_1:  -8.229423489585358
glc__D_e_1:  -9.308779104078127
strain_number:  0 12.578947368421053
strain_various:  0 6.081168425416497
strain_number:  1 9.210526315789474
strain_various:  1 3.532986620387034
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847, 9.466473132819145, 10.27272718554296, 7.633985046802309, 4.7799314551060315, 4.68231856744121, 4.631921010377828, 4.5083335971913225, 5.417892158052973, 6.758976106831042]
slip_r:  5.199888287978875
13
glc__D_e_

glc__D_e_1:  -0.578229415143888
strain_number:  0 2.736842105263158
strain_various:  0 0.7139294719079231
strain_number:  1 2.4210526315789473
strain_various:  1 0.8153649149910351
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847, 9.466473132819145, 10.27272718554296, 7.633985046802309, 4.7799314551060315, 4.68231856744121, 4.631921010377828, 4.5083335971913225, 5.417892158052973, 6.758976106831042, 4.382210140238313, 3.660927004161894, 2.6638005167631786, 4.506449514991181, 3.878611111111112]
slip_r:  3.818399657453136
18
glc__D_e_1:  -1.1091934898990217
glc__D_e_1:  -1.0835275741675139
glc__D_e_1:  -0.35849888442519773
glc__D_e_1:  -1.6130373182367168
glc__D_e_1:  -0.5415908352706194
glc__D_e_1:  -0.24609192868401197
glc__D_e_1:  -0.7603215553157288
glc__D_e_1:  -0.7045762861929699
glc__D_e_1:  -1.5307467923801268
glc__D_e_1:  -0.46879451982260667
glc__D_e_1:  -0.9450693737440428
glc__D_e_1:  -0.4277705682148041
glc__D_e_1:  -0.32520464354910494
glc__D_e_1:  -1.508108318

glc__D_e_1:  -0.21110134362330002
strain_number:  0 0.47368421052631576
strain_various:  0 0.5954583420518296
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847, 9.466473132819145, 10.27272718554296, 7.633985046802309, 4.7799314551060315, 4.68231856744121, 4.631921010377828, 4.5083335971913225, 5.417892158052973, 6.758976106831042, 4.382210140238313, 3.660927004161894, 2.6638005167631786, 4.506449514991181, 3.878611111111112, 4.159722222222223, 6.277777777777778, 5.361111111111111, 6.25, 7.75, 5.5, 0.25, 0.25, 2.0, 3.25]
slip_r:  2.25
28
glc__D_e_1:  -0.08983613624617698
glc__D_e_1:  -0.13870981470282606
strain_number:  0 0.3157894736842105
strain_various:  0 0.5668594533825794
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847, 9.466473132819145, 10.27272718554296, 7.633985046802309, 4.7799314551060315, 4.68231856744121, 4.631921010377828, 4.5

strain_number:  0 0.10526315789473684
strain_various:  0 0.4465937565388721
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847, 9.466473132819145, 10.27272718554296, 7.633985046802309, 4.7799314551060315, 4.68231856744121, 4.631921010377828, 4.5083335971913225, 5.417892158052973, 6.758976106831042, 4.382210140238313, 3.660927004161894, 2.6638005167631786, 4.506449514991181, 3.878611111111112, 4.159722222222223, 6.277777777777778, 5.361111111111111, 6.25, 7.75, 5.5, 0.25, 0.25, 2.0, 3.25, 4.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0]
slip_r:  0.4
42
strain_number:  0 0.10526315789473684
strain_various:  0 0.4465937565388721
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 366.2526672304941, 11.957224878582847, 9.466473132819145, 10.27272718554296, 7.633985046802309, 4.7799314551060315, 4.68231856744121, 4.631921010377828, 4.5083335971913225, 5.417892158052973, 6.758976106831042, 4.3822101

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 8, 33, 25, 34, 38, 23, 37, 27, 34, 22,  9,  6,  4, 24, 13, 16, 18,
       12, 30]), 1: array([36,  0, 17, 26, 17, 32, 37, 35, 16, 38, 23, 33, 36, 26, 21,  1,  8,
        6, 23])}
1
strain_number:  0 25.57894736842105
strain_various:  0 12.832179080132818
strain_number:  1 23.05263157894737
strain_various:  1 12.424397408708765
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 30.31578947368421
strain_various:  0 13.63066446841149
strain_number:  1 23.36842105263158
strain_various:  1 9.217891791829224
fd_r_threshold:  [1.05, 321.2030176751716]
slip_r:  65.08060353503433
3
strain_number:  0 35.89473684210526
strain_various:  0 13.43559583558034
strain_number:  1 23.526315789473685
strain_various:  1 8.987525981531183
fd_r_threshold:  [1.05, 321.2030176751716, 11.229438507513644]
slip_r:  67.11649123653704
4
strain_number:  0 42.73684210526316
strain_various:  0 15.011722104875796
strain_number:  1 23.736842105263158
strain_various:  1 8.07101444914003
fd_r_threshold:

glc__D_e_1:  -4.994356045040426
glc__D_e_1:  -3.7079043094047397
glc__D_e_1:  -2.289791672063366
glc__D_e_1:  -3.8207609818299635
glc__D_e_1:  -8.229599024541935
glc__D_e_1:  -5.296910348439639
glc__D_e_1:  -2.557639170849934
glc__D_e_1:  -3.5968159619507163
glc__D_e_1:  -6.974282066788146
glc__D_e_1:  -8.876756151514218
glc__D_e_1:  -3.4248648582306185
glc__D_e_1:  -1.5688782911056602
glc__D_e_1:  -6.40139156568285
glc__D_e_1:  -6.297843505111391
glc__D_e_1:  -0.969378666536906
glc__D_e_1:  -0.5331779016402418
glc__D_e_1:  -4.812524055119502
glc__D_e_1:  -6.869442561646368
glc__D_e_1:  -2.527444649129084
glc__D_e_1:  -2.0912438842324197
glc__D_e_1:  -6.306792419174847
glc__D_e_1:  -10.21528864919845
glc__D_e_1:  -6.70422594317801
glc__D_e_1:  -3.428453573824756
glc__D_e_1:  -8.935898703392114
glc__D_e_1:  -9.860809896084088
glc__D_e_1:  -5.291986688740886
glc__D_e_1:  -5.403170196279189
glc__D_e_1:  -13.614456745843473
glc__D_e_1:  -10.063990382538002
glc__D_e_1:  -2.994031253625987
g

glc__D_e_1:  -0.12973743660138792
glc__D_e_1:  -0.6439670632331045
glc__D_e_1:  -1.7649172944690825
glc__D_e_1:  -1.003367537700893
glc__D_e_1:  -1.5175971643326096
glc__D_e_1:  -1.2030356076942517
glc__D_e_1:  -1.2600404514444774
glc__D_e_1:  -3.1116181749412153
glc__D_e_1:  -0.5936170235618938
glc__D_e_1:  -1.585063646329056
glc__D_e_1:  -1.9448488511599786
glc__D_e_1:  -0.9831153912143855
glc__D_e_1:  -1.5216451963169466
glc__D_e_1:  -0.544082304282848
glc__D_e_1:  -1.0175972125538757
glc__D_e_1:  -0.6801574918713058
strain_number:  0 2.736842105263158
strain_various:  0 0.9647527778854399
strain_number:  1 2.0
strain_various:  1 0.9176629354822471
fd_r_threshold:  [1.05, 321.2030176751716, 11.229438507513644, 8.751385981876053, 9.042064304453397, 7.573319078251935, 4.914068952644876, 3.9184802664345466, 3.8715547569434703, 4.1099425794481395, 4.273224230324425, 5.194892237163168, 3.7876223407224794, 4.751675601636943, 4.006335209703717, 3.015538548752835, 4.262222222222222]
slip_r:

glc__D_e_1:  -0.13575741575326283
glc__D_e_1:  -0.171070289188481
strain_number:  0 0.3684210526315789
strain_various:  0 0.48237638894272
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 321.2030176751716, 11.229438507513644, 8.751385981876053, 9.042064304453397, 7.573319078251935, 4.914068952644876, 3.9184802664345466, 3.8715547569434703, 4.1099425794481395, 4.273224230324425, 5.194892237163168, 3.7876223407224794, 4.751675601636943, 4.006335209703717, 3.015538548752835, 4.262222222222222, 7.305555555555555, 6.777777777777778, 2.423611111111111, 6.361111111111111, 4.75, 4.5, 2.0, 6.25, 0.0, 2.0]
slip_r:  2.95
28
strain_number:  0 0.3684210526315789
strain_various:  0 0.48237638894272
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 321.2030176751716, 11.229438507513644, 8.751385981876053, 9.042064304453397, 7.573319078251935, 4.914068952644876, 3.9184802664345466, 3.8715547569434703, 4.1099425794481395, 4.273224230324425, 5.194892237163168, 3.7

strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 321.2030176751716, 11.229438507513644, 8.751385981876053, 9.042064304453397, 7.573319078251935, 4.914068952644876, 3.9184802664345466, 3.8715547569434703, 4.1099425794481395, 4.273224230324425, 5.194892237163168, 3.7876223407224794, 4.751675601636943, 4.006335209703717, 3.015538548752835, 4.262222222222222, 7.305555555555555, 6.777777777777778, 2.423611111111111, 6.361111111111111, 4.75, 4.5, 2.0, 6.25, 0.0, 2.0, 0.0, 2.0, 2.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0, 0, 0]
slip_r:  0.2
42
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 321.2030176751716, 11.229438507513644, 8.751385981876053, 9.042064304453397, 7.573319078251935, 4.914068952644876, 3.9184802664345466, 3.8715547569434703, 4.1099425794481395, 4.273224230324425, 5.194892237163168, 3.7876223407224794, 4.751675601636943, 4.006335209703717, 3.01

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 5,  4,  0, 29, 29, 30, 22,  8, 16,  3, 39, 36, 39, 23,  5, 10, 18,
       16, 27]), 1: array([ 3,  3, 10, 31, 16,  4, 33,  7, 29, 23, 30, 27, 10, 23,  3, 10, 12,
        8, 32])}
1
strain_number:  0 22.263157894736842
strain_various:  0 14.84247853207372
strain_number:  1 16.736842105263158
strain_various:  1 11.21954171717768
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 26.31578947368421
strain_various:  0 13.135138216085839
strain_number:  1 16.789473684210527
strain_various:  1 7.258961411636189
fd_r_threshold:  [1.05, 180.86080484301277]
slip_r:  37.01216096860255
3
strain_number:  0 31.31578947368421
strain_various:  0 10.89689782788974
strain_number:  1 16.789473684210527
strain_various:  1 6.45289773641857
fd_r_threshold:  [1.05, 180.86080484301277, 7.325936649062032]
slip_r:  38.267348298414966
4
strain_number:  0 37.1578947368421
strain_various:  0 12.105949637285736
strain_number:  1 16.789473684210527
strain_various:  1 5.453911656285848
fd_r_thresho

glc__D_e_1:  -19.141235697741983
glc__D_e_1:  -7.103347487844001
glc__D_e_1:  -5.335330968195852
glc__D_e_1:  -16.265949886382092
glc__D_e_1:  -8.405578160055786
glc__D_e_1:  -2.7981932902234474
glc__D_e_1:  -5.014697889628415
glc__D_e_1:  -11.519184139822318
glc__D_e_1:  -7.637641262371123
glc__D_e_1:  -3.3676044894038064
glc__D_e_1:  -5.539828083558646
glc__D_e_1:  -9.585351740423473
glc__D_e_1:  -6.969493862450427
glc__D_e_1:  -5.528597705013945
glc__D_e_1:  -1.5566596639401338
glc__D_e_1:  -1.612251417709285
glc__D_e_1:  -6.602820634962412
glc__D_e_1:  -7.167946622823463
glc__D_e_1:  -3.699607965699384
glc__D_e_1:  -2.335413917240241
glc__D_e_1:  -2.388381573103387
glc__D_e_1:  -1.4108186810692882
glc__D_e_1:  -2.0425339380170375
glc__D_e_1:  -3.573503247783635
glc__D_e_1:  -9.391696880720444
glc__D_e_1:  -7.641911879682375
glc__D_e_1:  -1.5294636207887207
glc__D_e_1:  -3.060432930555318
glc__D_e_1:  -3.6669497858199933
glc__D_e_1:  -1.8662684235525901
glc__D_e_1:  -4.5980056243946

glc__D_e_1:  -0.27877996614508227
glc__D_e_1:  -0.3850814000171492
glc__D_e_1:  -0.1423302992784503
glc__D_e_1:  -0.14539400048758888
glc__D_e_1:  -0.228632916395195
strain_number:  0 2.8421052631578947
strain_various:  0 1.1361596392064686
strain_number:  1 1.263157894736842
strain_various:  1 0.6359497880839249
fd_r_threshold:  [1.05, 180.86080484301277, 7.325936649062032, 8.38012272452991, 8.479146912088899, 9.466061420210682, 8.403153537921924, 4.752164397175829, 5.110343709945191, 5.056589400473908, 4.970305663460511, 3.519731438258879, 5.216285192577563, 3.5824415234201945, 6.140021527894905, 4.077483462813134, 5.243909550045913, 3.450141723356009]
slip_r:  4.498799557506031
19
glc__D_e_1:  -0.30492363781124476
glc__D_e_1:  -0.07413305284797
glc__D_e_1:  -0.4178249494559436
glc__D_e_1:  -1.337643153824373
glc__D_e_1:  -1.2823907526113598
glc__D_e_1:  -1.1279463308105657
glc__D_e_1:  -0.5154347904523968
glc__D_e_1:  -0.4963616140777003
glc__D_e_1:  -0.5557682179310145
glc__D_e_1: 

strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 180.86080484301277, 7.325936649062032, 8.38012272452991, 8.479146912088899, 9.466061420210682, 8.403153537921924, 4.752164397175829, 5.110343709945191, 5.056589400473908, 4.970305663460511, 3.519731438258879, 5.216285192577563, 3.5824415234201945, 6.140021527894905, 4.077483462813134, 5.243909550045913, 3.450141723356009, 8.116666666666667, 3.6458333333333335, 7.222222222222222, 6.0, 2.611111111111111, 5.75, 3.0, 3.0, 0.0, 1.0]
slip_r:  2.55
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 180.86080484301277, 7.325936649062032, 8.38012272452991, 8.479146912088899, 9.466061420210682, 8.403153537921924, 4.752164397175829, 5.110343709945191, 5.056589400473908, 4.970305663460511, 3.519731438258879, 5.216285192577563,

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 3, 31, 18, 29, 14, 29,  1, 29, 22, 14, 28, 27,  8, 20, 37,  8, 30,
        8, 32]), 1: array([38, 35, 35, 32, 39, 32, 32,  6,  5,  6, 12, 36,  1, 11, 38, 36, 21,
        5, 14])}
1
strain_number:  0 24.0
strain_various:  0 12.855471904785793
strain_number:  1 23.36842105263158
strain_various:  1 14.305724015658502
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 28.31578947368421
strain_various:  0 6.061549494268618
strain_number:  1 23.736842105263158
strain_various:  1 11.818104333085778
fd_r_threshold:  [1.05, 381.78973216721363]
slip_r:  77.19794643344272
3
strain_number:  0 33.578947368421055
strain_various:  0 4.8700566242648256
strain_number:  1 23.94736842105263
strain_various:  1 10.42503910991289
fd_r_threshold:  [1.05, 381.78973216721363, 6.941534221548356]
slip_r:  78.3762532777524
4
strain_number:  0 39.89473684210526
strain_various:  0 9.79739351397044
strain_number:  1 24.263157894736842
strain_various:  1 9.77021308426338
fd_r_threshold:  [1.05, 381

glc__D_e_1:  -9.816557515668817
glc__D_e_1:  -2.750073942623821
glc__D_e_1:  -1.822080659061342
glc__D_e_1:  -4.685459696403065
glc__D_e_1:  -11.423096541957506
glc__D_e_1:  -2.709486920055877
glc__D_e_1:  -1.7814936364933978
glc__D_e_1:  -4.212750911044286
glc__D_e_1:  -8.430135984669478
glc__D_e_1:  -3.691238212522249
glc__D_e_1:  -1.3434591267314753
glc__D_e_1:  -13.690221076294408
glc__D_e_1:  -12.145776858286467
glc__D_e_1:  -3.265402991588212
glc__D_e_1:  -7.255334894683886
glc__D_e_1:  -4.1880137102550155
glc__D_e_1:  -3.056006396420123
glc__D_e_1:  -4.213657595303494
glc__D_e_1:  -5.800218658839242
glc__D_e_1:  -16.307477271823434
glc__D_e_1:  -9.927870292987125
glc__D_e_1:  -2.142159877677109
glc__D_e_1:  -4.656714224775337
glc__D_e_1:  -13.307769211649724
glc__D_e_1:  -6.568377027982493
glc__D_e_1:  -2.006234254837419
glc__D_e_1:  -3.5372035646040163
glc__D_e_1:  -15.538037472814215
glc__D_e_1:  -9.158430493977907
glc__D_e_1:  -1.685294474406325
glc__D_e_1:  -3.21626378417292

glc__D_e_1:  -0.9454753370660582
glc__D_e_1:  -1.1320287482035267
glc__D_e_1:  -1.6462583748352433
glc__D_e_1:  -0.3813849715672426
glc__D_e_1:  -1.1140453180638938
glc__D_e_1:  -0.5016002101311476
glc__D_e_1:  -0.7374923222743108
glc__D_e_1:  -0.5830479004735168
glc__D_e_1:  -0.733867810340586
glc__D_e_1:  -2.344730392958901
glc__D_e_1:  -1.3671675009248025
glc__D_e_1:  -0.5190849528404227
glc__D_e_1:  -0.4568012211913146
strain_number:  0 3.3684210526315788
strain_various:  0 1.4220269564322416
strain_number:  1 1.5789473684210527
strain_various:  1 0.8775437895017404
fd_r_threshold:  [1.05, 381.78973216721363, 6.941534221548356, 5.708279820284387, 9.709846501645718, 12.38954429819346, 9.65664343976025, 6.410056578368613, 5.496286354768562, 4.092277002236393, 4.519555225820522, 4.210818291648086, 7.460301421623178, 3.939169341054799, 3.985354325978286, 4.490405013857394, 5.320963718820861]
slip_r:  5.039238764266903
18
glc__D_e_1:  -0.22586056962784107
glc__D_e_1:  -0.542407084270018

glc__D_e_1:  -0.09950821614561112
glc__D_e_1:  -0.25179765472604554
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 381.78973216721363, 6.941534221548356, 5.708279820284387, 9.709846501645718, 12.38954429819346, 9.65664343976025, 6.410056578368613, 5.496286354768562, 4.092277002236393, 4.519555225820522, 4.210818291648086, 7.460301421623178, 3.939169341054799, 3.985354325978286, 4.490405013857394, 5.320963718820861, 5.4625, 7.984999999999999, 4.151111111111112, 8.333333333333334, 3.75, 2.0, 4.75, 3.0, 6.0, 2.0]
slip_r:  3.55
28
glc__D_e_1:  -0.030023768368599324
strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 381.78973216721363, 6.941534221548356, 5.708279820284387, 9.709846501645718, 12.38954429819346, 9.65664343976025, 6.410056578368613, 5.496286354768562, 4.092277002236393, 4.519555225820522

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([34, 35,  1, 21, 11,  8, 38, 36, 20,  4, 38, 22, 19, 26, 29, 19, 36,
       12, 36]), 1: array([ 7, 28, 10, 36, 29, 35, 29,  0, 27, 34, 11,  2, 29, 32, 21, 17, 25,
       13, 19])}
1
strain_number:  0 27.68421052631579
strain_various:  0 14.186919963859028
strain_number:  1 21.473684210526315
strain_various:  1 11.26881314001113
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.8421052631579
strain_various:  0 9.863612863741245
strain_number:  1 21.57894736842105
strain_various:  1 7.727706744147691
fd_r_threshold:  [1.05, 773.8458205315914]
slip_r:  155.60916410631827
3
strain_number:  0 39.1578947368421
strain_various:  0 11.563386673347074
strain_number:  1 21.68421052631579
strain_various:  1 5.666639508924113
fd_r_threshold:  [1.05, 773.8458205315914, 11.811062925201128]
slip_r:  157.7613766913585
4
strain_number:  0 46.63157894736842
strain_various:  0 10.579078291936645
strain_number:  1 21.736842105263158
strain_various:  1 5.523558209239649
fd_r_threshold:

glc__D_e_1:  -7.620926362364122
glc__D_e_1:  -4.688237686261826
glc__D_e_1:  -4.707164681665072
glc__D_e_1:  -4.818348189203374
glc__D_e_1:  -9.335241012667744
glc__D_e_1:  -7.585456011629676
glc__D_e_1:  -4.255711425469154
glc__D_e_1:  -2.891517377010011
glc__D_e_1:  -7.214171971930707
glc__D_e_1:  -7.779297959791759
glc__D_e_1:  -2.4029891481888854
glc__D_e_1:  -2.950373420623852
glc__D_e_1:  -7.123352134776766
glc__D_e_1:  -9.02582621950284
glc__D_e_1:  -2.7093997067191458
glc__D_e_1:  -4.240369016485744
glc__D_e_1:  -6.970627937534754
glc__D_e_1:  -6.198405828530785
glc__D_e_1:  -3.1110185920043785
glc__D_e_1:  -3.658402864439345
glc__D_e_1:  -13.037551208197772
glc__D_e_1:  -10.310203315125605
glc__D_e_1:  -3.1086996144384695
glc__D_e_1:  -5.623253961536697
glc__D_e_1:  -9.639121937179667
glc__D_e_1:  -6.551988839276577
glc__D_e_1:  -5.01319940327174
glc__D_e_1:  -6.107967948141673
glc__D_e_1:  -6.017227706646472
glc__D_e_1:  -3.9076575007774808
glc__D_e_1:  -2.158025579547986
glc

glc__D_e_1:  -0.36889081639861854
glc__D_e_1:  -1.6886175650680175
glc__D_e_1:  -0.7110546730339189
glc__D_e_1:  -1.1472654396011746
glc__D_e_1:  -0.1776450755589034
glc__D_e_1:  -0.13473647069508488
glc__D_e_1:  -0.6489660973268014
glc__D_e_1:  -1.43975458638494
glc__D_e_1:  -0.9410441344519129
glc__D_e_1:  -0.11792566421860828
glc__D_e_1:  -0.7169192740825105
glc__D_e_1:  -0.5502327126403215
glc__D_e_1:  -0.3957882908395275
strain_number:  0 3.0
strain_various:  0 1.0760551736979407
strain_number:  1 1.368421052631579
strain_various:  1 0.581334790378277
fd_r_threshold:  [1.05, 773.8458205315914, 11.811062925201128, 10.768280514450035, 7.161472455922539, 7.38571860536486, 5.521035904249838, 4.920428320363157, 4.7762442191293, 4.838005522900042, 4.761827884494041, 7.560030306239275, 4.251581939464116, 3.4845059815324966, 3.660310689090451, 5.479155328798185, 5.040555555555556]
slip_r:  4.383221898888161
18
glc__D_e_1:  -0.525424208348205
glc__D_e_1:  -0.2504529608991659
glc__D_e_1:  -

strain_number:  0 0.10526315789473684
strain_various:  0 0.3068922049918579
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 773.8458205315914, 11.811062925201128, 10.768280514450035, 7.161472455922539, 7.38571860536486, 5.521035904249838, 4.920428320363157, 4.7762442191293, 4.838005522900042, 4.761827884494041, 7.560030306239275, 4.251581939464116, 3.4845059815324966, 3.660310689090451, 5.479155328798185, 5.040555555555556, 4.145833333333333, 6.971111111111111, 10.943333333333333, 5.25, 2.0, 5.25, 3.0, 2.0, 4.0, 3.0, 1.0]
slip_r:  2.6
29
glc__D_e_1:  -0.00978094581115746
strain_number:  0 0.0
strain_various:  0 0.0
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 773.8458205315914, 11.811062925201128, 10.768280514450035, 7.161472455922539, 7.38571860536486, 5.521035904249838, 4.920428320363157, 4.7762442191293, 4.838005522900042, 4.761827884494041, 7.560030306239275, 4.251581939464116, 3.4845059815324966, 3.660310689090451, 5.479155328798185, 5.

In [6]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.1)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([37, 30, 12, 17, 14, 38, 27, 37,  3, 38, 18, 34, 10,  7, 23, 16, 36,
        6, 36]), 1: array([ 7, 18, 11, 15, 19,  0, 17, 25, 33, 39, 33, 38,  9, 11, 37, 25, 31,
        5, 14])}
1
strain_number:  0 27.31578947368421
strain_various:  0 14.531463618749735
strain_number:  1 20.68421052631579
strain_various:  1 12.170078197123457
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 32.36842105263158
strain_various:  0 9.074026126174871
strain_number:  1 20.789473684210527
strain_various:  1 8.906407790068434
fd_r_threshold:  [1.05, 158.72617364250746]
slip_r:  32.58523472850149
3
strain_number:  0 38.36842105263158
strain_various:  0 7.463670088578121
strain_number:  1 20.842105263157894
strain_various:  1 7.492193536536524
fd_r_threshold:  [1.05, 158.72617364250746, 11.025167528117462]
slip_r:  34.580268234124986
4
strain_number:  0 45.68421052631579
strain_various:  0 10.848490710348642
strain_number:  1 20.94736842105263
strain_various:  1 7.351861090624936
fd_r_thresh

glc__D_e_1:  -8.9052936647523
glc__D_e_1:  -1.806116276254146
glc__D_e_1:  -2.646093931981068
glc__D_e_1:  -9.586780947071633
glc__D_e_1:  -14.402594255007974
glc__D_e_1:  -6.171409552674927
glc__D_e_1:  -3.762804604421479
glc__D_e_1:  -5.349365667957227
glc__D_e_1:  -3.146078621190724
glc__D_e_1:  -1.3453972589233207
glc__D_e_1:  -6.422079164505822
glc__D_e_1:  -8.9922252653732
glc__D_e_1:  -4.824457729186054
glc__D_e_1:  -4.206680041982878
glc__D_e_1:  -4.538435709311187
glc__D_e_1:  -5.141411735515305
glc__D_e_1:  -6.670345118675935
glc__D_e_1:  -4.560774912806943
glc__D_e_1:  -4.097079229568576
glc__D_e_1:  -4.700055255772694
glc__D_e_1:  -8.341089957280154
glc__D_e_1:  -7.41442342647539
glc__D_e_1:  -2.509256221007457
glc__D_e_1:  -3.0566404934424236
glc__D_e_1:  -9.064782499819838
glc__D_e_1:  -8.652345595646791
glc__D_e_1:  -3.775925743461369
glc__D_e_1:  -3.395316732333856
glc__D_e_1:  -10.332969979432274
glc__D_e_1:  -9.920533075259227
glc__D_e_1:  -4.107325496402366
glc__D_e_

glc__D_e_1:  -0.9898295746394496
glc__D_e_1:  -0.4196613651198784
glc__D_e_1:  -0.933890991751595
glc__D_e_1:  -0.26486580159303763
glc__D_e_1:  -1.8141898130110543
glc__D_e_1:  -0.8140655861586719
glc__D_e_1:  -1.8609938392076355
glc__D_e_1:  -0.945578442649869
glc__D_e_1:  -1.4663443164224559
glc__D_e_1:  -1.8261295212533786
glc__D_e_1:  -1.3611340093976387
glc__D_e_1:  -0.040735577781255694
glc__D_e_1:  -0.5549652044129725
glc__D_e_1:  -0.11715221016908961
strain_number:  0 2.473684210526316
strain_various:  0 0.6781104593013224
strain_number:  1 1.736842105263158
strain_various:  1 0.8486587103472158
fd_r_threshold:  [1.05, 158.72617364250746, 11.025167528117462, 6.8378879632360485, 8.354497559130134, 6.692980341839033, 4.391789167565296, 3.3530123072038354, 4.565987913881897, 5.27100164019188, 5.368258513716281, 4.732463273317101, 3.7982738797401745, 3.813702633946141, 3.7829324319259388, 4.455277777777778, 4.639722222222221]
slip_r:  4.097981789122451
18
glc__D_e_1:  -0.805646176

glc__D_e_1:  -0.09665046172043917
glc__D_e_1:  -0.08065987283045706
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 158.72617364250746, 11.025167528117462, 6.8378879632360485, 8.354497559130134, 6.692980341839033, 4.391789167565296, 3.3530123072038354, 4.565987913881897, 5.27100164019188, 5.368258513716281, 4.732463273317101, 3.7982738797401745, 3.813702633946141, 3.7829324319259388, 4.455277777777778, 4.639722222222221, 5.979166666666666, 4.673611111111111, 6.611111111111111, 4.361111111111111, 4.5, 2.5, 7.0, 2.0, 2.0, 0.0, 2.0]
slip_r:  2.6
29
strain_number:  0 0.21052631578947367
strain_various:  0 0.40768245749551757
strain_number:  1 0.05263157894736842
strain_various:  1 0.22329687826943606
fd_r_threshold:  [1.05, 158.72617364250746, 11.025167528117462, 6.8378879632360485, 8.354497559130134, 6.692980341839033, 4.391789167565296, 3.3530123072038354, 4.5

In [7]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.06)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([19, 16, 39, 24, 15, 36,  4, 11, 33, 18, 23, 25, 25,  0, 17,  0, 17,
       12, 19]), 1: array([16,  3, 33,  6, 24, 35,  2, 17, 36, 37, 19, 34, 34, 39, 32, 32, 26,
       18, 10])}
1
strain_number:  0 21.894736842105264
strain_various:  0 12.493765758388651
strain_number:  1 24.31578947368421
strain_various:  1 12.33332085547521
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 25.842105263157894
strain_various:  0 8.561008043785602
strain_number:  1 24.57894736842105
strain_various:  1 8.47482829805067
fd_r_threshold:  [1.05, 362.9780412038637]
slip_r:  73.43560824077274
3
strain_number:  0 30.63157894736842
strain_various:  0 9.674766562209188
strain_number:  1 24.894736842105264
strain_various:  1 8.06483423552756
fd_r_threshold:  [1.05, 362.9780412038637, 7.606967253859883]
slip_r:  74.74700169154471
4
strain_number:  0 36.36842105263158
strain_various:  0 10.193963777987697
strain_number:  1 25.05263157894737
strain_various:  1 7.401804077460308
fd_r_threshold:  

glc__D_e_1:  -5.330963432344821
glc__D_e_1:  -3.0489985074041126
glc__D_e_1:  -5.071760335836526
glc__D_e_1:  -11.95489724094605
glc__D_e_1:  -9.896223396306393
glc__D_e_1:  -5.203777273144311
glc__D_e_1:  -4.331375743350983
glc__D_e_1:  -5.821495742972117
glc__D_e_1:  -7.209740201066474
glc__D_e_1:  -2.7348425513074632
glc__D_e_1:  -2.7904343050766145
glc__D_e_1:  -5.89567986631528
glc__D_e_1:  -7.798153951041353
glc__D_e_1:  -6.984394877208043
glc__D_e_1:  -3.216829989188974
glc__D_e_1:  -4.442629886187997
glc__D_e_1:  -7.836896489579885
glc__D_e_1:  -4.620554081089558
glc__D_e_1:  -1.289189957967153
glc__D_e_1:  -7.165095006806586
glc__D_e_1:  -11.073591236830191
glc__D_e_1:  -5.436844639606501
glc__D_e_1:  -4.564443109813173
glc__D_e_1:  -10.977645195705007
glc__D_e_1:  -13.085460063461209
glc__D_e_1:  -8.344337634095947
glc__D_e_1:  -8.019320376737586
glc__D_e_1:  -12.987949834769985
glc__D_e_1:  -12.26662408699535
strain_number:  0 11.736842105263158
strain_various:  0 4.39906814

glc__D_e_1:  -3.0503362844691804
glc__D_e_1:  -0.5262780655748248
glc__D_e_1:  -1.4300225347691597
glc__D_e_1:  -3.1271558364651035
strain_number:  0 2.736842105263158
strain_various:  0 0.8486587103472158
strain_number:  1 2.4210526315789473
strain_various:  1 1.5666185332776017
fd_r_threshold:  [1.05, 362.9780412038637, 7.606967253859883, 8.10859729434104, 5.620116190652276, 6.10275850358524, 6.4421354180539945, 4.960097249847229, 4.518490981506239, 4.279120268104773, 4.7195453881866, 4.58451621363933, 3.6294493125045295, 4.139738804434733, 3.209162439692743, 2.9791504472159236, 4.215199829931973]
slip_r:  3.6345401667559805
18
glc__D_e_1:  -0.7169874225440578
glc__D_e_1:  -0.6079601022417678
glc__D_e_1:  -0.7508979072349558
glc__D_e_1:  -0.053930552217147154
glc__D_e_1:  -0.1733656596541766
glc__D_e_1:  -0.6875952862858932
glc__D_e_1:  -0.9832308824031151
glc__D_e_1:  -0.6079557205999244
glc__D_e_1:  -0.09894142547683105
glc__D_e_1:  -0.684687232623107
glc__D_e_1:  -0.71308784417060

strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 362.9780412038637, 7.606967253859883, 8.10859729434104, 5.620116190652276, 6.10275850358524, 6.4421354180539945, 4.960097249847229, 4.518490981506239, 4.279120268104773, 4.7195453881866, 4.58451621363933, 3.6294493125045295, 4.139738804434733, 3.209162439692743, 2.9791504472159236, 4.215199829931973, 7.660277777777778, 3.2083333333333335, 5.944444444444445, 3.5, 9.5, 2.75, 1.0, 5.0, 9.0, 0.0]
slip_r:  3.55
28
strain_number:  0 0.2631578947368421
strain_various:  0 0.44034738238635557
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 362.9780412038637, 7.606967253859883, 8.10859729434104, 5.620116190652276, 6.10275850358524, 6.4421354180539945, 4.960097249847229, 4.518490981506239, 4.279120268104773, 4.7195453881866, 4.58451621363933, 3.6294493125045295, 4.139738804434733, 3.209162439692743, 2.9791504472159236, 4.215199829931

In [8]:
distribute_micro_list_t, number, various = time_space_state([model0, model1], ['BIOMASS_Ec_iML1515_core_75p37M', 'BIOMASS_SC5_notrace'], [0.7074, 0.28], [5, 30], [[0.13, 10, 1268], [200, 50, 3410]], public_metabolism, vmax, km, 1/15, 0.1, 0.05, 2, 0.02)
print('The change process of microorganism distribution is: ',distribute_micro_list_t)
print('The mean numbers of microorganisms are: ', number)
print('The standard deviations of microorganisms are: ', various)

{0: array([ 2, 34, 17, 33, 33, 10, 18, 13, 29,  8, 35, 27, 17,  7,  7, 37, 19,
       23, 19]), 1: array([39, 10,  2, 29, 16, 34, 36, 15,  2, 37, 11, 39, 22, 38, 35, 36, 15,
       29,  3])}
1
strain_number:  0 24.0
strain_various:  0 12.769206380153285
strain_number:  1 24.0
strain_various:  1 13.745812759566189
fd_r_threshold:  [1.05]
slip_r:  1.05
2
strain_number:  0 28.31578947368421
strain_various:  0 10.000277004473675
strain_number:  1 24.105263157894736
strain_various:  1 7.099998049236984
fd_r_threshold:  [1.05, 75.61953015767553]
slip_r:  15.963906031535107
3
strain_number:  0 33.578947368421055
strain_various:  0 9.756594474204205
strain_number:  1 24.263157894736842
strain_various:  1 8.135963730079832
fd_r_threshold:  [1.05, 75.61953015767553, 26.116643907348656]
slip_r:  20.977234813004838
4
strain_number:  0 39.89473684210526
strain_various:  0 8.65657484740025
strain_number:  1 24.473684210526315
strain_various:  1 6.596397909007705
fd_r_threshold:  [1.05, 75.6195301576

glc__D_e_1:  -12.17034684134737
glc__D_e_1:  -8.105650851410182
glc__D_e_1:  -2.1781855163041897
glc__D_e_1:  -4.200947344736603
glc__D_e_1:  -10.417630143075685
glc__D_e_1:  -5.83870452650678
glc__D_e_1:  -3.649944757355485
glc__D_e_1:  -4.252920783559603
glc__D_e_1:  -6.6584960726037465
glc__D_e_1:  -5.217599915167266
glc__D_e_1:  -5.700953013234592
glc__D_e_1:  -7.779306595436156
glc__D_e_1:  -5.322413254068133
glc__D_e_1:  -4.035961518432446
glc__D_e_1:  -4.797159852425957
glc__D_e_1:  -3.4329658039668143
glc__D_e_1:  -12.197469007855624
glc__D_e_1:  -10.807469211648478
glc__D_e_1:  -4.261632036918067
glc__D_e_1:  -2.8974379884589245
glc__D_e_1:  -7.121613689278024
glc__D_e_1:  -11.030109919301628
glc__D_e_1:  -2.291697872900551
glc__D_e_1:  -2.3472896266697023
glc__D_e_1:  -5.4187726069466535
glc__D_e_1:  -8.14436516190603
glc__D_e_1:  -3.678729986888259
glc__D_e_1:  -1.8227434197633006
glc__D_e_1:  -6.909682263493272
glc__D_e_1:  -8.812156348219345
glc__D_e_1:  -4.895786675389716

glc__D_e_1:  -0.7900516793533418
glc__D_e_1:  -0.4449311933626823
glc__D_e_1:  -0.025740150405520357
glc__D_e_1:  -1.8773178739022582
glc__D_e_1:  -0.10091265619352763
glc__D_e_1:  -1.3347776777442095
glc__D_e_1:  -1.849007304375926
glc__D_e_1:  -0.09617411401530895
glc__D_e_1:  -1.2955280140198175
glc__D_e_1:  -1.8097576406515343
glc__D_e_1:  -1.398601810779997
glc__D_e_1:  -1.6138774510662333
glc__D_e_1:  -0.7631712084194668
glc__D_e_1:  -0.9746692882796397
glc__D_e_1:  -2.157572963343867
glc__D_e_1:  -0.09282799253829155
glc__D_e_1:  -0.8397308436162714
glc__D_e_1:  -1.353960470247988
strain_number:  0 3.263157894736842
strain_various:  0 1.1626695807565537
strain_number:  1 1.6842105263157894
strain_various:  1 0.7981974151633211
fd_r_threshold:  [1.05, 75.61953015767553, 26.116643907348656, 11.927255410458068, 11.337281419289276, 9.244712502369053, 4.976195176004279, 5.707528129299479, 5.114280601387396, 5.235263936486561, 4.906557126370806, 4.442898907889326, 5.091779312160026, 3

glc__D_e_1:  -0.045525352746468184
strain_number:  0 0.15789473684210525
strain_various:  0 0.36464227527765836
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 75.61953015767553, 26.116643907348656, 11.927255410458068, 11.337281419289276, 9.244712502369053, 4.976195176004279, 5.707528129299479, 5.114280601387396, 5.235263936486561, 4.906557126370806, 4.442898907889326, 5.091779312160026, 3.869299903246998, 5.737438271604938, 3.2702267573696155, 3.734019274376418, 4.658333333333333, 7.006944444444445, 6.833333333333334, 7.173611111111111, 1.7222222222222223, 6.0, 2.75, 1.0, 6.0, 3.0]
slip_r:  3.75
28
strain_number:  0 0.05263157894736842
strain_various:  0 0.22329687826943606
strain_number:  1 0.0
strain_various:  1 0.0
fd_r_threshold:  [1.05, 75.61953015767553, 26.116643907348656, 11.927255410458068, 11.337281419289276, 9.244712502369053, 4.976195176004279, 5.707528129299479, 5.114280601387396, 5.235263936486561, 4.906557126370806, 4.442898907889326, 5.091779312160